# TTM SEMANTIC4X3 — Huấn luyện 12 nhánh có resume (Colab Free T4)

**Owner-approved 2026-11-10.** Bốn nhánh PT-REAL/PT-PERM/NULL-REAL/NULL-PERM × ba seed S1/S2/S3, đúng **2048 optimizer updates/arm**. Một **StudyID duy nhất**; H768 đã niêm phong không được dùng trước khi cả 12 arm hoàn tất. Mã đã sửa chỉ là **giao thức phục hồi an toàn trước CLAIM**, không thay seeds/lịch TRAIN/kiến trúc/optimizer/điều kiện khoa học.

**Chạy lần đầu:** theo thứ tự Cell 1 → Cell 2 (PRE-CLAIM PASS) → Cell 3. **Khi Colab bị ngắt:** kết nối lại Free Tesla T4 → Cell 1 → Cell 2 (progress audit) → Cell 3 (tự chọn resume đúng CLAIM). Cell 4 chỉ xem tiến độ, không huấn luyện. **Không chạy đồng thời hai Colab session. Không nhấn Run all.**

`MAX_CHUNKS_PER_INVOCATION=0` để chạy liên tục cho tới khi Colab ngắt hoặc đủ 12 arm. Có thể đặt `4` để dừng có chủ đích sau 4 checkpoint (mỗi checkpoint 256 update) rồi gọi Cell 3 lại; không rút ngắn tổng huấn luyện. Nếu ngắt giữa 256 update, resume làm lại chỉ đoạn chưa có READY từ checkpoint và RNG đã khóa, ghi biên bản replay (không thêm update khoa học). Nếu bị ngắt trong lúc **đang ghi binary checkpoint** và file chưa có READY, runner sẽ HOLD để giữ nguyên bằng chứng; không tự xóa.

Đã khóa: `STUDY_ID=ttm_gemma2_semantic4x3_3seed_2048x4096_h768_20261110_v2`, H768 768 truyện/1000 cửa sổ/186655 target, `L0019` bất biến. Từng nhánh GPU chạy **tuần tự** vì chỉ có 1 T4; CUDA đã tự chạy nhiều luồng/kernels nội bộ.

In [ ]:
# CELL 1 — stage/verify exactly existing Drive sources; upgrade old pre-CLAIM runner only.
# NEVER overwrite source when scientific CLAIM already exists.
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ.setdefault('OMP_NUM_THREADS','2')
os.environ.setdefault('OPENBLAS_NUM_THREADS','2')
from pathlib import Path
import hashlib, zipfile
from google.colab import drive

drive.mount('/content/drive')
PILOT=Path('/content/drive/MyDrive/TTM/ttm-small-pilot')
OLD=PILOT/'TTM_GEMMA2_CAUSAL6_INPUT_PRIVATE_V1_20261109.zip'
HZIP=PILOT/'TTM_SEMANTIC4X3_H768_PRIVATE_V1_20261110.zip'
SOURCE_BUNDLE=PILOT/'TTM_SEMANTIC4X3_SCIENCE_RESUME_SOURCE_20261110.zip'
SRC=Path('/content/ttm_semantic4x3_source_v2')
INP=SRC/'input';H768=INP/'h768';CODE=SRC/'code'
STUDY_ID='ttm_gemma2_semantic4x3_3seed_2048x4096_h768_20261110_v2'
RUN=PILOT/STUDY_ID
OLD_RUNNER_SHA='9a824e453195cafc530e2289d3b8a87e354faf8905ca73b2f29b79144d657a8d'
NEW_RUNNER_SHA='a3182549739cda848ce0f7f449b1f1455e06419d72295dcee96f0e8a7822f676'
source_files={
 'ttm_semantic4x3_runner.py':NEW_RUNNER_SHA,
 'ttm_semantic4x3_independent_readback.py':'51732bd62cfd13bf8956f11ee862b939d0dc78a62c58cd3d655dc05b566764e2',
 'ttm_semantic4x3_balanced_null_cpu_qa.py':'0d71f38da356056be50a18d36f589774190178fd321e6d2e3970413d96238a2d'
}
input_files={
 'checkpoint.pt':'cb985751795c8328ad65acf7ca4c1033ce35182c1bfbdb8c38550de629cb271b',
 'candidate_initialization.pt':'c9fc798d8880c7e76636258052e9885dbb086d7b462a5b6bee5cff8b582973c3',
 'T4_PRIVATE_TRAIN_VAL_WINDOWS.npz':'184f68b4c0c800ef3e98362c2e20e1cd35baa269e926489a475664526ecd9913',
 'T4_PRIVATE_PACK_MANIFEST.json':'ef2e6e67d49b1d5a921b1051df329f15574f1c10cece0ce49468acb53322d650',
 'src/p02_exact.py':'e0fcac285918b66b13c8f694e6d625a0691c9b59221b43043e63f46b25ddcc07',
 'src/ttm_embeddinggemma2_swap.py':'d11da61e9d7bb040f3263e680f026817a513553c186ab243e184ffaf6e68cd39',
 'src/ttm_directswap2048_gpu_once.py':'0994f0cd91cb504f2db8478f55963813a85469807f2921e38c196248f323759e'
}
h768_files={
 'H768_PUBLIC_SEAL.json':'9725b1254cde81b77e23b85ab479a08cafc14af0a7a536c7722ca87c7f2a5aa0',
 'H768_MEMBERSHIP_PRIVATE.json':'0ab671ae59ff4300d8257f2689261d6328afd7563c0aa66b363bc22348c1628f',
 'H768_WINDOWS_PRIVATE.npz':'1eb5b54b4bb7744e23fffd0fd755d5c91853a773dcc0d2219f777b47ba6a53df'
}

def sha_file(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as fd:
        for v in iter(lambda:fd.read(1<<20),b''):h.update(v)
    return h.hexdigest()
def sha_bytes(b):return hashlib.sha256(b).hexdigest()
def entries(archive):
    with zipfile.ZipFile(archive) as z:
        info={}
        for v in z.infolist():
            if v.is_dir():continue
            normalized=v.filename.replace('\\','/')
            assert normalized not in info,'HOLD_DUPLICATE_ZIP_ENTRY'
            info[normalized]=v
        assert z.testzip() is None,'HOLD_CORRUPT_ZIP'
        return {k:z.read(i) for k,i in info.items()}
def stage(archive,expected,prefix,target,allow_extra):
    values=entries(archive);wanted={prefix+name for name in expected}
    assert (wanted.issubset(values) if allow_extra else wanted==set(values)), 'HOLD_ZIP_WHITELIST'
    for name,sh in expected.items():
        data=values[prefix+name]
        assert sha_bytes(data)==sh,'HOLD_ZIP_SOURCE_SHA_'+name
        dest=target/name;dest.parent.mkdir(parents=True,exist_ok=True)
        if dest.exists():
            assert sha_file(dest)==sh,'HOLD_STAGE_EXISTING_SOURCE_DRIFT_'+name
        else:
            with dest.open('xb') as f:f.write(data)
        assert sha_file(dest)==sh,'HOLD_STAGED_FILE_SHA_'+name

assert OLD.is_file() and HZIP.is_file() and SOURCE_BUNDLE.is_file(),'HOLD_REQUIRED_DRIVE_ZIP_MISSING'
assert sha_file(OLD)=='2c6cb72f35eba72d6014f15914d6e9792cbf0bf6774d93e3ff5c53e5c6b9ba9a','HOLD_OLD_ZIP_SHA'
assert sha_file(HZIP)=='27825597aac0fa33753704f681343bc09211144590f475928aa812271ff609b1','HOLD_H768_ZIP_SHA'
assert sha_file(SOURCE_BUNDLE)=='67f07792446387b6c27ba5c334509b4ac11d3a3e7faf74ea44f329715eeae035','HOLD_NEW_PUBLIC_BUNDLE_SHA'

INP.mkdir(parents=True,exist_ok=True)
stage(OLD,input_files,'input/',INP,allow_extra=True)
H768.mkdir(parents=True,exist_ok=True)
stage(HZIP,h768_files,'h768/',H768,allow_extra=False)
CODE.mkdir(parents=True,exist_ok=True)
runner=CODE/'ttm_semantic4x3_runner.py'
if runner.exists() and sha_file(runner)==OLD_RUNNER_SHA:
    assert not RUN.exists(),'HOLD_OLD_SOURCE_WITH_STUDY_CLAIM_DO_NOT_PATCH'
    runner_data=entries(SOURCE_BUNDLE)['ttm_semantic4x3_runner.py']
    assert sha_bytes(runner_data)==NEW_RUNNER_SHA
    runner.write_bytes(runner_data)
    print('OLD_PRECLAIM_RUNNER_UPGRADED_BEFORE_CLAIM=YES')
stage(SOURCE_BUNDLE,source_files,'',CODE,allow_extra=True)
assert sha_file(runner)==NEW_RUNNER_SHA
print('SEMANTIC4X3_RESUMABLE_SOURCE_SHA256=PASS')
print('H768_PRIVATE_SHA3=PASS_768_STORIES_NOT_SCORED')
print('STUDY_CLAIM_EXISTS=',(RUN/'STUDY_CLAIM.json').exists())


In [ ]:
# CELL 2 — Exactly one integrated new-source preflight, or read-only progress for resume.
from google.colab import auth
import subprocess,sys
from pathlib import Path

OWNER_VERIFIED_FREE_GIB=9.0 # Owner attested approximately 10GB decimal, checked against real Drive API by runner.
OWNER_CONFIRMS_COLAB_FREE_T4=True
assert OWNER_CONFIRMS_COLAB_FREE_T4
assert sha_file(CODE/'ttm_semantic4x3_runner.py')==NEW_RUNNER_SHA
runner=CODE/'ttm_semantic4x3_runner.py'
common=[sys.executable,'-B',str(runner),
    '--input',str(INP),'--h768',str(H768),'--code-dir',str(CODE),
    '--drive-output',str(RUN),'--owner-verified-free-gib',str(OWNER_VERIFIED_FREE_GIB),
    '--free-colab-confirmed']

if RUN.exists():
    assert (RUN/'STUDY_CLAIM.json').is_file(),'HOLD_STUDY_DIRECTORY_WITHOUT_CLAIM'
    print('EXISTING_STUDY=READ_ONLY_RESUME_AUDIT')
    args=common+['--mode','progress']
else:
    auth.authenticate_user()  # Explicit Google API auth for Drive quota check.
    args=common+['--mode','preflight']

r=subprocess.run(args,text=True,capture_output=True,check=False)
print('=== STDOUT ===');print(r.stdout[-16000:])
print('=== STDERR ===');print(r.stderr[-2500:])
print('PREFLIGHT_OR_AUDIT_EXIT_CODE=',r.returncode)
assert r.returncode==0,'HOLD_NEW_SOURCE_PREFLIGHT_OR_RESUME_AUDIT'
if not RUN.exists():
    assert 'TTM_SEMANTIC4X3_PRECLAIM=PASS_NO_SCIENCE_CONSUMED' in r.stdout
    assert not RUN.exists(), 'HOLD_CLAIM_PREMATURE'
    print('SEMANTIC4X3_NEW_SOURCE_PRECLAIM=PASS_NO_SCIENCE_CONSUMED')
else:
    print('SEMANTIC4X3_RESUME_AUDIT=PASS_READ_ONLY')
PRECHECK_NEW_SOURCE_PASS=True


In [ ]:
# CELL 3 — Owner-authorized science execution / exact resume on same StudyID.
# This cell creates ONE CLAIM only if no original run exists. Never run two Colab sessions in parallel.
import subprocess,sys,json
from pathlib import Path

OWNER_SCIENCE_APPROVED=True  # Approval explicitly given 2026-11-10, 12 arms × 2048 steps, Free T4.
MAX_CHUNKS_PER_INVOCATION=0   # 0=continuous; 4=graceful pause after 4 verified READY chunks.
assert PRECHECK_NEW_SOURCE_PASS and OWNER_SCIENCE_APPROVED
assert MAX_CHUNKS_PER_INVOCATION==0 or 1<=MAX_CHUNKS_PER_INVOCATION<=96
assert sha_file(CODE/'ttm_semantic4x3_runner.py')==NEW_RUNNER_SHA

if RUN.exists():
    assert (RUN/'STUDY_CLAIM.json').is_file(),'HOLD_PARTIAL_STUDY_ROOT_NO_CLAIM'
    assert not (RUN/'RESULT.json').exists(),'HOLD_SCIENCE_ALREADY_HAS_RESULT'
    mode='resume'
else:
    mode='execute'
print('SCIENCE_STUDY_ID=',STUDY_ID)
print('SCIENCE_MODE=',mode)
print('OWNER_SCIENCE_APPROVAL=GRANTED_FOR_ORIGINAL_12_ARMS')
args=common+['--mode',mode,'--owner-science-approved',
    '--max-chunks-per-invocation',str(MAX_CHUNKS_PER_INVOCATION)]
if mode=='resume':
    # Runner itself verifies original CLAIM SHA, source, environment, all prior READY, and the journal.
    args+=['--manual-resume-adjudicated']
result=subprocess.run(args,check=False)
print('SEMANTIC4X3_RUNNER_EXIT_CODE=',result.returncode)
if result.returncode!=0:
    print('HOLD_FAILED_OR_INTERRUPTED_RETAIN_ALL_EVIDENCE_DO_NOT_DELETE_OR_CHANGE_STUDY_ID')
else:
    if (RUN/'RESULT.json').is_file():
        print('SEMANTIC4X3_SCIENCE_COMPLETED_PENDING_INDEPENDENT_QA')
    else:
        print('SEMANTIC4X3_CONTINUE_SAME_STUDY_BY_RERUNNING_CELLS_2_AND_3')


In [ ]:
# CELL 4 — READ-ONLY progress inspection. No GPU/model/score/CLAIM writes.
assert sha_file(CODE/'ttm_semantic4x3_runner.py')==NEW_RUNNER_SHA
p=subprocess.run(common+['--mode','progress'],text=True,capture_output=True,check=False)
print(p.stdout[-12000:]);print(p.stderr[-1000:]);print('PROGRESS_EXIT=',p.returncode)
if p.returncode!=0:print('HOLD_READONLY_AUDIT_EXCEPTION_KEEP_EVIDENCE')
